# 07 · Claims Guardrail (AI4)

**Question:** Can a retrieval-augmented checker flag non-compliant green and labelling claims (recall ≥ 0.9) while never citing a clause that isn't in the source text?

**How it works:** keyword rules → hybrid retrieval (BM25 + embeddings, fused) → cross-encoder rerank → LLM verdict as fixed JSON → word-for-word check of the quoted clause. A failed quote check blocks the answer and sends the claim to a person. Autonomy level 2: the guardrail recommends; you decide.

**Pass test (brief 6A.1):** recall ≥ 0.9 on your 40 hand-labelled test claims; precision reported; zero invented clauses.

> TARU is a fictional brand created for an independent student portfolio project. Market, competitor and regulatory facts come from public sources as cited. Consumer findings come from the author's own research with a convenience sample. Costs, targets and financial projections are illustrative assumptions.

## 1. Setup (Colab)
Set your GitHub username. The knowledge base (`ai/kb/chunks.jsonl`, 143 clauses incl. GOTS v8.0) is in the repo, so the regulation PDFs are **not** needed to run this notebook.

In [ ]:
GITHUB_USER = "<your-username>"          # ← change
DRIVE_KB = "/content/drive/MyDrive/taru-category-launch/kb_raw"
DRIVE_OUT = "/content/drive/MyDrive/taru-category-launch/ai_outputs"

import os, sys
if not os.path.exists("/content/taru-category-launch"):
    !git clone -q https://github.com/{GITHUB_USER}/taru-category-launch.git /content/taru-category-launch
%cd /content/taru-category-launch
!pip install -q rank_bm25 chromadb sentence-transformers pypdf scikit-learn transformers accelerate bitsandbytes
sys.path.insert(0, "/content/taru-category-launch")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:          # Drive is optional: outputs then stay in /content/ai_outputs
    print("Drive not mounted:", e); DRIVE_OUT = "/content/ai_outputs"
!mkdir -p "{DRIVE_OUT}"
# ai/kb/chunks.jsonl is already in the repo; section 3 can be skipped if the PDFs are not in Drive

## 2. Model
The verdict step uses **Qwen2.5-7B-Instruct**, an open-weight model (Apache 2.0), loaded in 4-bit so it fits Colab's free T4 GPU. No API key, no rate limit, and no claim text leaves the notebook.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. The first load downloads about 15 GB (5–10 minutes).

The pipeline is model-agnostic: to run the same evaluation on an API model, set `PROVIDER = "gemini"` or `"anthropic"` and store the key in Colab Secrets.

In [ ]:
import os, importlib, pandas as pd, torch
PROVIDER = "local"                          # "local" (no key), "gemini" or "anthropic"
MODEL = "Qwen/Qwen2.5-7B-Instruct"          # API examples: "gemini-2.0-flash", "claude-haiku-4-5-20251001"
PRICE_IN_PER_M, PRICE_OUT_PER_M = 0.0, 0.0  # local model: no per-token price

if PROVIDER == "local":
    assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then rerun from the top"
    print("GPU:", torch.cuda.get_device_name(0))
else:
    from google.colab import userdata
    key_name = "GEMINI_API_KEY" if PROVIDER == "gemini" else "ANTHROPIC_API_KEY"
    os.environ[key_name] = userdata.get(key_name)

from ai import guardrail as g
importlib.reload(g)
llm = g.make_llm(PROVIDER, MODEL)
text, tin, tout = llm("Reply with JSON only.", 'Return {"ok": true}')   # one test call
print(f"Using {MODEL}. Test call OK ({tin} in / {tout} out tokens): {text.strip()[:60]}")

## 3. Load the knowledge base
Loads the pre-built clauses from the repo. Set `REBUILD = True` only if you have put every PDF in `ai/kb/raw/` under the names in `ai/kb/README.md`; any `MISSING` row means that document would drop out.

In [ ]:
REBUILD = False
if REBUILD:
    chunks, report = g.build_chunks(); display(pd.DataFrame(report).T)
    assert not any(v["status"] == "MISSING" for k, v in report.items() if k != "GOTS-LABEL"), "A PDF is missing: fix names or set REBUILD = False"
    g.save_chunks(chunks)
else:
    chunks = g.load_chunks()
print(f"{len(chunks)} clause chunks:", pd.Series([c.doc_id for c in chunks]).value_counts().to_dict())

**Spot-check:** compare these five against the PDFs. The clause number and the text must match the source exactly.

In [ ]:
import random
random.seed(7)
for c in random.sample(chunks, min(5, len(chunks))):
    print(f"[{c.key}] p.{c.page}\n{c.text[:400]}\n")

## 4. Retrieval index and one test claim

In [ ]:
retriever = g.Retriever(chunks)           # downloads the embedding and reranker models on first run
guard = g.Guardrail(retriever, llm, MODEL, g.load_prompt(), prompt_version="guardrail_v1")

demo = guard.check("Our kurtas are eco-friendly and made from natural fibres.", "product page")
assert demo.status != "error", f"Call failed: {demo.reason}"
pd.Series(vars(demo))

## 5. Gold test set (your 40 claims)
Export the `Guardrail_Test_Claims` tab from the data-collection Sheet as CSV and save it to Drive as `taru-category-launch/ai_outputs/guardrail_test_claims.csv`.

**DL-04:** the labels must be complete before any claim is sent to the model. The cell below stops if any label is missing.

In [ ]:
gold = pd.read_csv("ai/eval/guardrail_test_claims.csv")   # in the repo
gold.columns = [c.strip().lower().split(" (")[0].replace(" ", "_").rstrip("?") for c in gold.columns]
# keep only the claim rows (the Sheet tab also holds the category list further down)
gold = gold[gold["claim_id"].astype(str).str.match(r"GC\d+") & gold["claim_text"].notna()].reset_index(drop=True)
assert gold["your_label"].isin(["compliant", "non-compliant"]).all(), "Label every claim before running the model"
print(len(gold), "claims ·", gold["your_label"].value_counts().to_dict())

from sklearn.metrics import cohen_kappa_score
both = gold[gold["classmate_label"].isin(["compliant", "non-compliant"])]
if len(both):
    print(f"Human agreement (you vs classmate): kappa = {cohen_kappa_score(both['your_label'], both['classmate_label']):.2f} on {len(both)} claims")

## 6. Run the guardrail on the gold set

In [ ]:
results = []
for i, row in gold.iterrows():
    r = guard.check(row["claim_text"], str(row.get("where_it_would_appear", "")))
    results.append(r)
    if i == 2 and all(x.status == "error" for x in results):
        raise RuntimeError(f"First 3 calls failed: {r.reason}. Fix MODEL/key in section 2; nothing has been scored.")

out = pd.DataFrame([{**{k: v for k, v in vars(r).items() if k not in ("retrieved",)},
                     "flagged": r.flagged, "retrieved": "; ".join(r.retrieved)} for r in results])
out.insert(0, "claim_id", gold["claim_id"]); out.insert(1, "your_label", gold["your_label"])
out.to_csv("ai/eval/guardrail_results_v1.csv", index=False)
print("status counts:", out["status"].value_counts().to_dict())
out[["claim_id", "your_label", "verdict", "risk_level", "status", "doc_id", "clause_id", "flagged"]]

## 7. Metrics
Positive class = non-compliant. A claim counts as **flagged** if the verdict is non-compliant or insufficient basis, or if the answer was blocked or failed: in every such case it goes to you.

`valid_run` must be True (at most 5% failed calls). A failed or unparseable answer is sent to you, so it counts as flagged, which would otherwise inflate recall.

In [ ]:
m = g.evaluate(gold.to_dict("records"), results)
m["cost_usd"] = m["total_input_tokens"] / 1e6 * PRICE_IN_PER_M + m["total_output_tokens"] / 1e6 * PRICE_OUT_PER_M
m["cost_per_1000_claims_usd"] = m["cost_usd"] / max(m["n"], 1) * 1000
display(pd.Series(m))
assert m["valid_run"], f"{m['parse_or_call_errors']} of {m['n']} calls failed: these metrics are not a result. Fix and rerun section 6."
assert m["invented_clauses_passed"] == 0

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(gold["your_label"], ["non-compliant" if r.flagged else "compliant" for r in results],
                                       labels=["non-compliant", "compliant"], cmap="Greens")
plt.title("Guardrail v1 vs your labels"); plt.show()

per_cat = out.assign(category=gold["category"]).groupby(["category", "your_label"])["flagged"].mean().rename("share flagged")
display(per_cat.to_frame())

## 8. Ablation: keyword rules as escalation
Reapplies the rules to the saved results (no new API calls): any high-risk rule hit turns a `compliant` verdict into `non-compliant`. Report both rows; use `escalate` only if recall in step 7 is below 0.9 (brief section 15).

In [ ]:
import copy
def escalate(r):
    r = copy.copy(r)
    if r.verdict == "compliant" and any(h["risk"] == "high" for h in r.rule_hits):
        r.verdict, r.risk_level = "non-compliant", "high"
    return r
m_esc = g.evaluate(gold.to_dict("records"), [escalate(r) for r in results])
pd.DataFrame([m, m_esc], index=["flag_only", "escalate"])[["recall_noncompliant", "precision_noncompliant", "fp", "fn"]]

## 9. Red-team (10 attacks)
`expected_flagged` is checked automatically; also read the reason and rewrite for RT01, RT02, RT07 and RT08.

In [ ]:
rt = pd.read_csv("ai/eval/redteam_guardrail.csv").fillna("")
rt_res = [guard.check(c, "red-team") for c in rt["input_claim"]]
rt["flagged"] = [r.flagged for r in rt_res]
rt["status"] = [r.status for r in rt_res]
rt["verdict"] = [r.verdict for r in rt_res]
rt["reason"] = [r.reason for r in rt_res]
rt["rewrite"] = [r.compliant_rewrite for r in rt_res]
expect_error = rt["check"].str.contains("status = error", regex=False)   # RT09: an empty input SHOULD return an error
rt["auto_pass"] = ((rt["status"] != "error") | expect_error) & (rt["flagged"] == rt["expected_flagged"].astype(str).str.upper().eq("TRUE"))
print(f"{rt['auto_pass'].sum()} of {len(rt)} attacks handled as expected;", (rt["status"] == "error").sum(), "call errors")
rt.to_csv("ai/eval/redteam_guardrail_results_v1.csv", index=False)
rt[["attack_id", "attack_type", "expected_flagged", "flagged", "status", "auto_pass", "reason", "rewrite"]]

## 10. Pre-screen the claims matrix (D7)
Run once D7 exists. Export its sheet as CSV to `ai_outputs/claims_matrix.csv` with a `claim_wording` column. Every row gets a guardrail verdict; you remain the decision-maker.

In [ ]:
import os
d7_path = "compliance/claims_matrix.csv"
if os.path.exists(d7_path):
    d7 = pd.read_csv(d7_path)
    d7_res = [guard.check(w, "claims matrix") for w in d7["claim_wording"].fillna("")]
    d7["guardrail_verdict"] = [r.verdict for r in d7_res]
    d7["guardrail_risk"] = [r.risk_level for r in d7_res]
    d7["guardrail_clause"] = [f"{r.doc_id} {r.clause_id}" if r.status == "ok" else r.status for r in d7_res]
    d7.to_csv("compliance/claims_matrix_prescreened.csv", index=False); d7.to_csv(f"{DRIVE_OUT}/claims_matrix_prescreened.csv", index=False)
    display(d7)
else:
    print("D7 not built yet")

## 11. Save outputs to Drive

In [ ]:
!mkdir -p "{DRIVE_OUT}" && cp ai/eval/*results*.csv ai/logs/calls.csv ai/kb/chunks.jsonl "{DRIVE_OUT}/" && ls "{DRIVE_OUT}"

## Answer

*Write one paragraph once the run is complete: recall and precision on the 40 test claims, how many answers were blocked for unverified citations, whether keyword escalation was needed, red-team results, average seconds per claim on a T4 GPU, and what the guardrail still misses.*